In [1]:
import nflreadpy as nfl
import pandas as pd
# Load weekly player stats for 2019-2025
# 2018 used only as historical context for 2019 features
player_stats = nfl.load_player_stats(range(2018,2026))

# Convert from Polars to pandas
df = player_stats.to_pandas()

# Only use regular-season games for fantasy modeling
df = df[df['season_type'] == 'REG'].copy()



In [2]:
# Player Metadata Merging

players = nfl.load_players().to_pandas()

player_metadata = players[
    ['gsis_id', 'rookie_season']
].copy()

player_metadata = player_metadata.rename(
    columns={'gsis_id': 'player_id'}
)

df = df.merge(
    player_metadata,
    on='player_id',
    how='left'
)


In [3]:
# Player Season Status Merging

player_seasons = (
    df[
        ['player_id', 'season']
    ]
    .dropna(subset=['player_id'])
    .drop_duplicates()
    .sort_values(['player_id', 'season'])
)

player_seasons['last_active_season'] = (
    player_seasons
    .groupby('player_id')['season']
    .shift(1)
)

player_seasons['season_gap'] = (
    player_seasons['season']
    - player_seasons['last_active_season']
)

df = df.merge(
    player_seasons,
    on=['player_id', 'season'],
    how='left'
)

df['is_rookie'] = (
    df['season'] == df['rookie_season']
).astype(int)

df['is_veteran_after_gap'] = (
    (df['season'] > df['rookie_season']) &
    (
        df['last_active_season'].isna() |
        (df['season_gap'] > 1)
    )
).astype(int)

# Sorting dataset into players and seasons (chronologically)
df = df.sort_values(['player_id','season','week']).reset_index(drop=True)



In [4]:
# Early Season Baseline Helper

def create_early_season_baseline(df, stat, positions):

    # Only use columns needed for calculating baseline
    baseline_df = df[
        [
            'player_id',
            'season',
            'position',
            'last_active_season',
            'is_rookie',
            'is_veteran_after_gap'
        ]
    ].copy()

    # Keep original row order
    baseline_df['row_id'] = df.index

    # Veterans

    # Average stat for each player's season
    player_season_avg = (
        df[df['position'].isin(positions)]
        .groupby(
            ['player_id', 'season'],
            as_index=False
        )[stat]
        .mean()
        .rename(
            columns={
                'season': 'last_active_season',
                stat: 'veteran_last_active_season_avg'
            }
        )
    )

    baseline_df = baseline_df.merge(
        player_season_avg,
        on=['player_id', 'last_active_season'],
        how='left',
        validate='many_to_one'
    )

    # Rookies

    rookie_historical_avg_rows = []

    for season in range(2019,2026):

        # Only use rookies from seasons before the current season
        historical_rookies = df[
            (df['season'] < season) &
            (df['is_rookie'] == 1) &
            (df['position'].isin(positions))
        ]

        position_historical_avg = (
            historical_rookies
            .groupby('position')[stat]
            .mean()
        )

        for position, historical_avg in position_historical_avg.items():
            rookie_historical_avg_rows.append({
                'season': season,
                'position': position,
                'rookie_historical_avg': historical_avg
            })
    
    rookie_historical_avg = pd.DataFrame(
        rookie_historical_avg_rows
    )

    baseline_df = baseline_df.merge(
        rookie_historical_avg,
        on=['season', 'position'],
        how='left',
        validate='many_to_one'
    )

    # Historical Position Average 
    position_historical_avg_rows = []

    for season in range(2019,2026):
        
        # Only use seasons before current season
        historical_players = df[(df['season'] < season) & (df['position'].isin(positions))]

        position_historical_avg = (historical_players.groupby('position')[stat].mean())

        for position, historical_avg in position_historical_avg.items():
            position_historical_avg_rows.append({
                'season': season,
                'position': position,
                'position_historical_avg': historical_avg
            })
    
    position_historical_avg_df = pd.DataFrame(position_historical_avg_rows)

    baseline_df = baseline_df.merge(
        position_historical_avg_df,
        on=['season', 'position'],
        how='left',
        validate='many_to_one'
    )

    # Choosing Baseline

    baseline_df['baseline'] = float('nan')

    # Rookies use historical average for rookies at their position
    baseline_df.loc[
        baseline_df['is_rookie'] == 1,
        'baseline'
    ] = baseline_df.loc[
        baseline_df['is_rookie'] == 1,
        'rookie_historical_avg'
    ]

    # Veterans after a season gap use their last active-season average
    baseline_df.loc[
        baseline_df['is_veteran_after_gap'] == 1,
        'baseline'
    ] = baseline_df.loc[
        baseline_df['is_veteran_after_gap'] == 1,
        'veteran_last_active_season_avg'
    ]

    # If veteran's last active season is outside dataset,
    # Use historical average for their position

    missing_veteran_baseline = (
        (baseline_df['is_veteran_after_gap'] == 1) &
        (baseline_df['baseline'].isna())
    )

    baseline_df.loc[
        missing_veteran_baseline,
        'baseline'
    ] = baseline_df.loc[
        missing_veteran_baseline,
        'position_historical_avg'
    ]

    # Restore original DF order
    baseline_df = baseline_df.sort_values('row_id')

    return baseline_df['baseline'].set_axis(df.index)

In [5]:
# Rolling Average Helper

# Calculate average stat from up to the previous N eligible games
def rolling_avg_with_last_season(df, stat, window, baseline=None):

    past_stats = []

    for games_back in range(1, window + 1):

        # Get player's stat from N games ago
        previous_value = (
            df.groupby('player_id')[stat]
            .shift(games_back)
        )

        # Get season that previous game occurred in
        previous_season = (
            df.groupby('player_id')['season']
            .shift(games_back)
        )

        # Only allow games from current season or immediately previous season
        valid_history = (
            (previous_season == df['season']) |
            (previous_season == df['season'] - 1)
        )

        # Keep the stat only if it comes from valid recent history
        eligible_value = previous_value.where(
            valid_history
        )

        # If no valid recent game exists, use the player's baseline
        if baseline is not None:
            eligible_value = eligible_value.fillna(
                baseline
            )

        past_stats.append(
            eligible_value
        )

    # Put previous game stats side-by-side and average across them
    return pd.concat(
        past_stats,
        axis=1
    ).mean(axis=1)


In [6]:
# ----------- WR/TE Feature Engineering -----------
receiving_positions = ['RB', 'WR', 'TE']

receiving_stats = [
    'targets',
    'receptions',
    'receiving_yards',
    'target_share',
    'receiving_air_yards',
    'air_yards_share',
    'receiving_tds',
    'receiving_yards_after_catch'
]

receiving_feature_columns = {}

# Create 3 game and 5 game averages
for stat in receiving_stats:

    baseline = create_early_season_baseline(
        df,
        stat,
        receiving_positions
    )

    avg_3 = rolling_avg_with_last_season(
        df,
        stat,
        3,
        baseline
    )

    avg_5 = rolling_avg_with_last_season(
        df,
        stat,
        5,
        baseline
    )

    receiving_feature_columns[f'{stat}_baseline'] = baseline
    receiving_feature_columns[f'{stat}_avg_3'] = avg_3
    receiving_feature_columns[f'{stat}_avg_5'] = avg_5

receiving_features_df = pd.DataFrame(
    receiving_feature_columns,
    index=df.index
)

df = pd.concat(
    [df, receiving_features_df],
    axis=1
)

# Trends
receiving_trend_columns = {
    'targets_trend':
        df['targets_avg_3'] - df['targets_avg_5'],

    'target_share_trend':
        df['target_share_avg_3'] - df['target_share_avg_5'],

    'rec_yards_trend':
        df['receiving_yards_avg_3'] - df['receiving_yards_avg_5'],

    'rec_air_yards_trend':
        df['receiving_air_yards_avg_3'] - df['receiving_air_yards_avg_5']
}

df = pd.concat(
    [
        df,
        pd.DataFrame(
            receiving_trend_columns,
            index=df.index
        )
    ],
    axis=1
)


In [7]:
# ----------- RB Feature Engineering -----------

df['opportunities'] = df['carries'] + df['targets']

rb_stats = [
    'carries',
    'rushing_yards',
    'rushing_tds',
    'opportunities'
]

rb_feature_columns= {}

# Create 3 game and 5 game averages
for stat in rb_stats:

    baseline = create_early_season_baseline(
        df,
        stat,
        ['RB']
    )

    avg_3 = rolling_avg_with_last_season(
        df,
        stat,
        3,
        baseline
    )

    avg_5 = rolling_avg_with_last_season(
        df,
        stat,
        5,
        baseline
    )

    rb_feature_columns[f'rb_{stat}_baseline'] = baseline
    rb_feature_columns[f'rb_{stat}_avg_3'] = avg_3
    rb_feature_columns[f'rb_{stat}_avg_5'] = avg_5

rb_features_df = pd.DataFrame(
    rb_feature_columns,
    index=df.index
)

df = pd.concat(
    [df, rb_features_df],
    axis=1
)

# Trends
rb_trend_columns = {
    'rb_carries_trend':
        df['rb_carries_avg_3'] - df['rb_carries_avg_5'],

    'rb_rushing_yards_trend':
        df['rb_rushing_yards_avg_3'] - df['rb_rushing_yards_avg_5'],

    'rb_opportunities_trend':
        df['rb_opportunities_avg_3'] - df['rb_opportunities_avg_5']
}

df = pd.concat(
    [
        df,
        pd.DataFrame(
            rb_trend_columns,
            index=df.index
        )
    ],
    axis=1
)

In [8]:
# ----------- QB Feature Engineering -----------

qb_passing_stats = [
    'completions',
    'attempts',
    'passing_yards',
    'passing_tds',
    'passing_interceptions',
    'passing_air_yards',
    'passing_first_downs'
]

qb_rushing_stats = [
    'carries',
    'rushing_yards',
    'rushing_tds'
]

qb_feature_columns = {}

# Passing features
for stat in qb_passing_stats:

    baseline = create_early_season_baseline(
        df,
        stat,
        ['QB']
    )

    avg_3 = rolling_avg_with_last_season(
        df,
        stat,
        3,
        baseline
    )

    avg_5 = rolling_avg_with_last_season(
        df,
        stat,
        5,
        baseline
    )

    qb_feature_columns[f'{stat}_baseline'] = baseline
    qb_feature_columns[f'{stat}_avg_3'] = avg_3
    qb_feature_columns[f'{stat}_avg_5'] = avg_5


# QB rushing features
for stat in qb_rushing_stats:

    baseline = create_early_season_baseline(
        df,
        stat,
        ['QB']
    )

    avg_3 = rolling_avg_with_last_season(
        df,
        stat,
        3,
        baseline
    )

    avg_5 = rolling_avg_with_last_season(
        df,
        stat,
        5,
        baseline
    )

    qb_feature_columns[f'qb_{stat}_baseline'] = baseline
    qb_feature_columns[f'qb_{stat}_avg_3'] = avg_3
    qb_feature_columns[f'qb_{stat}_avg_5'] = avg_5


# Add all QB features at once
qb_features_df = pd.DataFrame(
    qb_feature_columns,
    index=df.index
)

df = pd.concat(
    [df, qb_features_df],
    axis=1
)

qb_trend_columns = {
    'attempts_trend':
        df['attempts_avg_3'] - df['attempts_avg_5'],

    'passing_yards_trend':
        df['passing_yards_avg_3'] - df['passing_yards_avg_5'],

    'passing_air_yards_trend':
        df['passing_air_yards_avg_3'] - df['passing_air_yards_avg_5'],

    'qb_carries_trend':
        df['qb_carries_avg_3'] - df['qb_carries_avg_5'],

    'qb_rushing_yards_trend':
        df['qb_rushing_yards_avg_3'] - df['qb_rushing_yards_avg_5']
}

df = pd.concat(
    [
        df,
        pd.DataFrame(
            qb_trend_columns,
            index=df.index
        )
    ],
    axis=1
)

In [9]:
# ----------- K Feature Engineering -----------

kicker_stats = [
    'fg_att',
    'fg_made',
    'fg_long',
    'fg_made_50_59',
    'pat_att',
    'pat_made'
]

kicker_feature_columns = {}

for stat in kicker_stats:

    baseline = create_early_season_baseline(
        df,
        stat,
        ['K']
    )

    avg_3 = rolling_avg_with_last_season(
        df,
        stat,
        3,
        baseline
    )

    avg_5 = rolling_avg_with_last_season(
        df,
        stat,
        5,
        baseline
    )

    kicker_feature_columns[f'{stat}_baseline'] = baseline
    kicker_feature_columns[f'{stat}_avg_3'] = avg_3
    kicker_feature_columns[f'{stat}_avg_5'] = avg_5

kicker_features_df = pd.DataFrame(
    kicker_feature_columns,
    index=df.index
)

df = pd.concat(
    [df, kicker_features_df],
    axis=1
)

# Trends
# Trends
kicker_trend_columns = {
    'fg_att_trend':
        df['fg_att_avg_3'] - df['fg_att_avg_5'],

    'fg_made_trend':
        df['fg_made_avg_3'] - df['fg_made_avg_5']
}

df = pd.concat(
    [
        df,
        pd.DataFrame(
            kicker_trend_columns,
            index=df.index
        )
    ],
    axis=1
)

# Kicker fantasy points
df['kicker_fantasy_points'] = (
    3 * df['fg_made'] +
    df['pat_made']
)

print(
    df[
        (df['season'] == 2020) &
        (df['week'] == 1) &
        (df['player_display_name'].isin([
            'Graham Gano',
            'Rodrigo Blankenship',
            'Tyler Bass'
        ]))
    ][
        [
            'player_display_name',
            'is_rookie',
            'is_veteran_after_gap',
            'fg_att_baseline',
            'fg_att_avg_3',
            'fg_att_avg_5',
            'fg_made_baseline',
            'fg_made_avg_3',
            'fg_made_avg_5'
        ]
    ].to_string(index=False)
)

player_display_name  is_rookie  is_veteran_after_gap  fg_att_baseline  fg_att_avg_3  fg_att_avg_5  fg_made_baseline  fg_made_avg_3  fg_made_avg_5
        Graham Gano          0                     1         1.333333      1.333333      1.333333          1.166667       1.166667       1.166667
Rodrigo Blankenship          1                     0         1.807407      1.807407      1.807407          1.474074       1.474074       1.474074
         Tyler Bass          1                     0         1.807407      1.807407      1.807407          1.474074       1.474074       1.474074


In [10]:
# ----------- Opponent Matchup Feature Engineering -----------

fantasy_positions = ['QB', 'RB', 'WR', 'TE', 'K']

matchup_df = df[
    df['position'].isin(fantasy_positions)
].copy()

matchup_df['matchup_points'] = matchup_df['fantasy_points_ppr']

matchup_df.loc[
    matchup_df['position'] == 'K',
    'matchup_points'
] = matchup_df.loc[
    matchup_df['position'] == 'K',
    'kicker_fantasy_points'
]

# Total fantasy points allowed by each defense to each position each week
weekly_points_allowed = (
    matchup_df.groupby(
        ['season', 'week', 'opponent_team', 'position'],
        as_index=False
    )['matchup_points']
    .sum()
)

weekly_points_allowed = weekly_points_allowed.rename(
    columns = {
        'matchup_points': 'points_allowed'
    }
)

# Sort each defense-position history chronologically
weekly_points_allowed = weekly_points_allowed.sort_values(
    ['opponent_team', 'position', 'season', 'week']
).reset_index(drop=True)

# Calculate points allowed using current and immediately previous season
def rolling_matchup_avg(
    df,
    stat,
    window
):

    past_stats = []

    matchup_group = [
        'opponent_team',
        'position'
    ]

    for games_back in range(1, window + 1):

        previous_value = (
            df.groupby(matchup_group)[stat]
            .shift(games_back)
        )

        previous_season = (
            df.groupby(matchup_group)['season']
            .shift(games_back)
        )

        # Only allow current season or immediately previous season
        valid_history = (
            (previous_season == df['season']) |
            (previous_season == df['season'] - 1)
        )

        past_stats.append(
            previous_value.where(valid_history)
        )

    return pd.concat(
        past_stats,
        axis=1
    ).mean(axis=1)

weekly_points_allowed['opp_points_allowed_avg_3'] = (
    rolling_matchup_avg(
        weekly_points_allowed,
        'points_allowed',
        3
    )
)

weekly_points_allowed['opp_points_allowed_avg_5'] = (
    rolling_matchup_avg(
        weekly_points_allowed,
        'points_allowed',
        5
    )
)

weekly_points_allowed['opp_points_allowed_trend'] = (
    weekly_points_allowed['opp_points_allowed_avg_3']
    - weekly_points_allowed['opp_points_allowed_avg_5']
)

matchup_features = weekly_points_allowed[
    [
        'season',
        'week',
        'opponent_team',
        'position',
        'opp_points_allowed_avg_3',
        'opp_points_allowed_avg_5',
        'opp_points_allowed_trend'
    ]
]

df = df.merge(
    matchup_features,
    on=[
        'season',
        'week',
        'opponent_team',
        'position'
    ],
    how='left'
)


In [11]:
# Remove context-only season before modeling
df = df[df['season'] >= 2019].copy()

# To Parquet
df.to_parquet(
    "../data/processed/player_features.parquet",
    index=False
)